# 0. 세팅하기

## 0. 설치하기

In [ ]:
!pip install duckdb

In [ ]:
!pip install python-dotenv

## 1. 세팅하기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [ ]:
import duckdb as dd
import dotenv
import os
import pandas as pd

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
# 한글 폰트 지정
import matplotlib.font_manager as fm

font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'  # 경로 확인 필요
fontprop = fm.FontProperties(fname=font_path)

plt.rc('font', family=fontprop.get_name())

In [ ]:
# 컬럼 모두 출력
pd.set_option('display.max_columns', None)

In [ ]:
# .env 파일 경로 찾기

env_path = dotenv.find_dotenv()

In [ ]:
# .env 파일 불러오기  (내용이 있으면 -> True, 없으면 -> False)

dotenv.load_dotenv(
    dotenv_path=env_path,
    override=True
)

In [ ]:
mem_con = dd.connect("mydb_copy.duckdb")

In [ ]:
HMAC_ID = os.getenv("HMAC_ID")
HMAC_PW = os.getenv("HMAC_PW")

secret_gcs = f"""
CREATE SECRET (
    TYPE GCS,
    KEY_ID '{HMAC_ID}',
    SECRET '{HMAC_PW}'
);
"""

mem_con.execute(secret_gcs)

In [ ]:
mem_con.execute("FROM duckdb_secrets()").df()

## 2. 파일 불러오기

### 1) 기존데이블

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/accounts_timelinereport.parquet')
"""

timelinereport_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/accounts_paymenthistory.parquet')
"""

paymenthistory_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/accounts_failpaymenthistory.parquet')
"""

failpaymenthistory_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/event_receipts.parquet')
"""

event_receipts_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/accounts_userwithdraw.parquet')
"""

userwithdraw_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/accounts_userquestionrecord.parquet')
"""

userquestionrecord_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/accounts_attendance.parquet')
"""

attendance_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/accounts_pointhistory.parquet')
"""

pointhistory_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/polls_questionpiece.parquet')
"""

polls_questionpiece_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/polls_questionset.parquet')
"""

polls_questionset_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/polls_question.parquet')
"""

polls_question_df = mem_con.execute(_query).df()

### 2) 병합테이블

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/accounts_paymenthistory.parquet')
"""

paymenthistory_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/question_df.parquet')
"""

question_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/vote_point_df.parquet')
"""

vote_point_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/event_info_df.parquet')
"""

event_info_df = mem_con.execute(_query).df()

In [ ]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/user_df.parquet')
"""

user_df = mem_con.execute(_query).df()

### 3) duckdb에 저장

In [ ]:
# mem_con.execute("""
#                 CREATE TABLE user_df AS
#                 SELECT * FROM user_df
#                 """)

In [ ]:
# mem_con.execute("""
#                 CREATE TABLE vote_point_df AS
#                 SELECT * FROM vote_point_df
#                 """)

### 4) duckdb 확인

In [ ]:
tables = mem_con.execute("SHOW TABLES").fetchall()
print(tables)

# 2. 피쳐엔지니어링

### 2) 질문테이블 : 질문piece, 질문set, 질문 테이블 병합

questionset 1정규화

In [ ]:
polls_questionset_df.head()

In [ ]:
print("shape : ", polls_questionpiece_df.shape)
print("id 갯수 : ", polls_questionpiece_df['id'].nunique())

In [ ]:
# 정규화 대상 DataFrame
df = polls_questionset_df.copy()

# 리스트 컬럼 정규화 (explode 사용)
df['question_piece_id_list'] = df['question_piece_id_list'].apply(eval)  # 문자열로 저장된 리스트를 진짜 리스트로 변환
normalized_df = df.explode('question_piece_id_list')[['id', 'question_piece_id_list']]

# 열 이름 바꾸기 (선택사항)
normalized_df = normalized_df.rename(columns={'question_piece_id_list': 'question_piece_id'})

# 결과 확인
normalized_df.head()

In [ ]:
merged_df = normalized_df.merge(
    df[['id', 'opening_time', 'status', 'created_at', 'user_id']],
    how='left',
    on='id'
)

In [ ]:
# 정규화 완료 테이블
merged_df.head()

In [ ]:
# question_piece_id 중복값 확인
merged_df['question_piece_id'].duplicated().sum()

In [ ]:
merged_df['question_piece_id'].nunique()

questionpiece 테이블 합치기

In [ ]:
polls_questionpiece_df.head()

In [ ]:
merged_df.head()

In [ ]:
question_df = merged_df.merge(
    polls_questionpiece_df, 
    how='left', 
    left_on='question_piece_id', 
    right_on='id',
    suffixes=('_set','_piece'))

In [ ]:
question_df.head(20)

In [ ]:
question_df.isna().sum()

In [ ]:
nan_rows = question_df[question_df.isna().any(axis=1)]
nan_rows

In [ ]:
nan_rows['status'].value_counts()

polls_question_df랑 merge

In [ ]:
polls_question_df.head()

In [ ]:
question_df2 = question_df.merge(
  polls_question_df,
  how='left',
  left_on='question_id',
  right_on='id'
)

In [ ]:
question_df2.head()

null 제거

In [ ]:
question_df2.isna().sum()

In [ ]:
question_df_notnull = question_df2.dropna()
question_df_notnull

In [ ]:
question_df_notnull.shape

columns 정리

In [ ]:
question_df_notnull.columns

In [ ]:
final_question_df = question_df_notnull[['id_set', 'question_piece_id', 'opening_time', 'status',
                                          'created_at_set', 'user_id', 'is_voted', 'is_skipped', 'created_at_piece',
                                          'question_id',  'question_text']]

In [ ]:
final_question_df

추출하기

In [ ]:
# mem_con.execute("""
#                COPY final_question_df TO 'gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/question_df.parquet' (FORMAT PARQUET);
#                """)

### 3) 투표, 포인트사용이력 outer join

In [ ]:
userquestionrecord_df.head(1)

In [ ]:
pointhistory_df.head(1)

In [ ]:
print(userquestionrecord_df.shape)
print(pointhistory_df.shape)

In [ ]:
vote_point_merge = userquestionrecord_df.merge(
    pointhistory_df[['delta_point', 'created_at', 'user_id', 'user_question_record_id']],
    how="outer",
    left_on="id",
    right_on="user_question_record_id"
)

In [ ]:
vote_point_merge.head()

In [ ]:
merge_df = vote_point_merge[['created_at_x', 'id',	'status',		'chosen_user_id',	'question_id',	'user_id_x',
                  'question_piece_id',	'has_read',	'answer_status',	'answer_updated_at',	'report_count',
                  'opened_times',	'delta_point',	'created_at_y',	'user_id_y']]

merge_df.head()

In [ ]:
# 조건 정의
conditions = [
    merge_df["user_id_y"] == merge_df["user_id_x"],
    merge_df["user_id_y"] == merge_df["chosen_user_id"]
]

# 각 조건에 해당하는 값
choices = ["chooser", "chosen"]

# 새 컬럼 추가
merge_df["point_user"] = np.select(conditions, choices, default=None)

In [ ]:
merge_df[merge_df['point_user'] == 'chosen'].head()

In [ ]:
merge_df = merge_df.drop(columns=['user_id_y'])

In [ ]:
merge_df.head()

In [ ]:
merge_df.shape

In [ ]:
merge_df.isna().sum()

In [ ]:
merge_df.dropna(subset=['created_at_x'], inplace=True)

In [ ]:
merge_df.isna().sum()

In [ ]:
# # 1. 임시 뷰 생성
# mem_con.register('temp_merge_df', merge_df)

# # 2. 뷰를 테이블로 변환
# mem_con.execute("CREATE TABLE vote_point_df_ver2 AS SELECT * FROM temp_merge_df")

# # 3. 임시 뷰 삭제 (선택사항)
# mem_con.execute("DROP VIEW temp_merge_df")

In [ ]:
vote_point_df_ver2 = mem_con.execute("SELECT * FROM vote_point_df_ver2").df()
vote_point_df_ver2.head()

## 4) 통합유저테이블 생성

### 1. 리스트 형식 숫자로 넣기

In [ ]:
import json

def fast_len(x):
    if x == '[]' or pd.isna(x):
        return 0
    try:
        return len(json.loads(x))
    except:
        return 0

user_df['friend_count'] = user_df['friend_id_list'].apply(fast_len)
user_df['block_user_id_count'] = user_df['block_user_id_list'].apply(fast_len)
user_df['hide_user_id_count'] = user_df['hide_user_id_list'].apply(fast_len)
user_df['attendance_count'] = user_df['attendance_date_list'].apply(fast_len)

In [ ]:
user_df.head()

In [ ]:
user_df['friend_id_list'][1]

In [ ]:
# 1. 임시 뷰 생성
mem_con.register('temp_user_df', user_df)

# 2. 뷰를 테이블로 변환
mem_con.execute("CREATE TABLE user_df_ver2 AS SELECT * FROM temp_user_df")

# 3. 임시 뷰 삭제 (선택사항)
mem_con.execute("DROP VIEW temp_user_df")

In [ ]:
mem_con.execute("SHOW TABLES").fetchall()

### 2. 학교 관련 컬럼 정리

In [ ]:
query = """
    SELECT * EXCLUDE (friend_id_list, block_user_id_list, hide_user_id_list, attendance_date_list, id_attendance)
    FROM user_df_ver2
    """
    
user_df_ver2 = mem_con.execute(query).df()

In [ ]:
user_df_ver2.head(5)

In [ ]:
user_df_ver2.columns

In [ ]:
columns = ['created_at', 'user_id', 'is_superuser', 'is_staff', 'gender', 'point', 'is_push_on',
       'ban_status', 'report_count', 'alarm_count','friend_count',
       'block_user_id_count', 'hide_user_id_count', 'attendance_count',
       'pending_chat', 'pending_votes', 'group_id', 'grade', 'class_num',
       'school_id', 'address', 'student_count', 'school_type', ]

In [ ]:
copy_df = user_df_ver2[columns]
user_df_ver3 = copy_df.copy()
user_df_ver3.head()

In [ ]:
user_df_ver3['school_grade'] = user_df_ver3.apply(
    lambda row: f"{row['school_type']}_{int(row['grade'])}" if pd.notna(row['grade']) else None,
    axis=1
)


In [ ]:
user_df_ver3.drop(columns=['group_id', 'grade', 'class_num', 'school_id', 'school_type'], inplace=True)

In [ ]:
user_df_ver3.head()

### 3. 포인트 사용/적립횟수 테이블 만들기

In [ ]:
pointhistory_df.head()

In [ ]:
use_df = pointhistory_df[pointhistory_df['delta_point'] < 0]
get_df = pointhistory_df[pointhistory_df['delta_point'] > 0]

In [ ]:
use_user_df = use_df['user_id'].value_counts().reset_index(name='pointuse_cnt')
use_user_df

In [ ]:
get_user_df = get_df['user_id'].value_counts().reset_index(name='pointget_cnt')
get_user_df

### 4. 투표참여 횟수

In [ ]:
userquestionrecord_df.head(3)

In [ ]:
vote_df = userquestionrecord_df['user_id'].value_counts().reset_index(name='vote_cnt')
vote_df

In [ ]:
chosen_df = userquestionrecord_df['chosen_user_id'].value_counts().reset_index(name='chosen_cnt')
chosen_df.rename(columns = {'chosen_user_id':'user_id'}, inplace=True)
chosen_df

### 5. 이벤트 참여

In [ ]:
event_receipts_df.head()

In [ ]:
event_df = event_receipts_df['user_id'].value_counts().reset_index(name='event_cnt')
event_df

### 6. 구매 관련 횟수

In [ ]:
paymenthistory_df.head()

In [ ]:
buy_df = paymenthistory_df['user_id'].value_counts().reset_index(name='buy_cnt')
buy_df

In [ ]:
failpaymenthistory_df.head()

In [ ]:
buyfail_df = failpaymenthistory_df['user_id'].value_counts().reset_index(name='buyfail_cnt')
buyfail_df

### 7. 친구 신고횟수

In [ ]:
timelinereport_df.head()

In [ ]:
friendreport_df = timelinereport_df['user_id'].value_counts().reset_index(name='friendreport_cnt')
friendreport_df

### 8. 데이터 합치기

In [ ]:
# 기준 DataFrame 복사
merged_df = user_df_ver3.copy()

# 기존 컬럼 목록 저장
original_columns = set(merged_df.columns)

# 병합 대상 DataFrame 리스트
df_list = [use_user_df, get_user_df, vote_df, chosen_df, event_df, buy_df, buyfail_df, friendreport_df]

# merge 실행
for df in df_list:
    merged_df = merged_df.merge(df, on='user_id', how='left')

# 새로 생긴 컬럼만 골라서 fillna(0)
new_columns = list(set(merged_df.columns) - original_columns)
merged_df[new_columns] = merged_df[new_columns].fillna(0)


In [ ]:
merged_df.head()

In [ ]:
merged_df.shape

In [ ]:
merged_df.isna().sum()

In [ ]:
merged_df.describe()

### 9. 결측치 등 제거

In [ ]:
# 필터링 조건 (제외할 조건을 반대로 표현)
filtered_df = merged_df[
    (merged_df['is_superuser'] != 1) &
    (merged_df['is_staff'] != 1) &
    (merged_df['gender'].notna()) &
    (merged_df['school_grade'] != 'None') &
    (merged_df['address'].notna()) &
    (merged_df['student_count'].notna())
]

In [ ]:
print(merged_df.shape)
print(filtered_df.shape)

address, student_count, school_grade도 none이 들어가거나 포함되면 지울까?

In [ ]:
filtered_df.head(5)

In [ ]:
filtered_df.columns

컬럼(is_superuser, is_staff) 제거

In [ ]:
columns = ['created_at', 'user_id', 'gender', 'point', 'is_push_on', 'ban_status', 'report_count', 'alarm_count',
       'friend_count', 'block_user_id_count', 'hide_user_id_count', 'attendance_count', 'pending_chat', 'pending_votes', 'address',
       'student_count', 'school_grade', 'pointuse_cnt', 'pointget_cnt', 'vote_cnt', 'chosen_cnt', 'event_cnt', 'buy_cnt', 'buyfail_cnt',
       'friendreport_cnt']

In [ ]:
final_user_df = filtered_df[columns]
final_user_df.head()

### 10. 추출하기

In [ ]:
# mem_con.execute("""
#                 COPY final_user_df TO 'gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/final_user_df.parquet' (FORMAT PARQUET);
#                 """)

# 3. EDA

### 1) 회원분포 확인

In [ ]:
user_df.head()

In [ ]:
user_df.describe()

In [ ]:
# 학교/학년별 유저수
school_grade_df = user_df.groupby('school_type')['grade'].value_counts().reset_index()

# 소계 만들기
subtotals = (
    school_grade_df.groupby('school_type')['count']
    .sum()
    .reset_index()
    .assign(grade='소계')
)

# 총계 만들기
grand_total = pd.DataFrame({
    'school_type': ['총계'],
    'grade': ['총계'],
    'count': [school_grade_df['count'].sum()]
})

# 합치기
final_df = pd.concat([school_grade_df, subtotals, grand_total], ignore_index=True)

# 총합 가져오기
total_count = final_df.loc[(final_df['school_type'] == '총계') & (final_df['grade'] == '총계'), 'count'].values[0]

# percentage 컬럼 추가
final_df['percentage(%)'] = round((final_df['count'] / total_count) * 100, 2)

final_df


In [ ]:
plot_df = final_df[
    (final_df['grade'] != '소계') & 
    (final_df['school_type'] != '총계')
]

plt.figure(figsize=(5,3))
sns.barplot(data=plot_df, x='school_type', y='count', hue='grade', order=['M', 'H'])

plt.title('School Type별 학년 분포')
plt.xlabel('Grade')
plt.ylabel('Count')
plt.legend(title='School Type')
plt.tight_layout()
plt.show()

In [ ]:
# 중학교, 고등학교 등록된 학교 수
user_df.groupby('school_type')['school_id'].nunique()

In [ ]:
# 등록된 학교 수
user_df['school_id'].nunique()

In [ ]:
user_df['year'] = user_df['created_at'].dt.year
user_df['month'] = user_df['created_at'].dt.month
user_df['year_month'] = user_df['year'].astype('str') + '_' + user_df['month'].astype('str')

In [ ]:
user_df['year_month'].value_counts().reset_index()

### 1-2) 회원 포인트 보유현황

(이상치 제거) point 4857.5 미만의 값만 추출

In [ ]:
# point 상한값 설정
q1 = user_df['point'].quantile(0.25)
q3 = user_df['point'].quantile(0.75)

iqr = q3 - q1 

print(q3 + 1.5*iqr)

In [ ]:
user_simple_table = mem_con.execute("""
                            SELECT user_id,	is_superuser,	is_staff,	gender,	point, grade, school_type
                            FROM user_df
                            WHERE point < 4857.5
                            """).df()
user_simple_table.head()

In [ ]:
print("point 상한선 제한 전 : ", user_df.shape)
print("point 상한선 제한 후 : ", user_simple_table.shape)
print("차이 : ", user_df.shape[0] - user_simple_table.shape[0], "(", round((user_df.shape[0] - user_simple_table.shape[0])/user_df.shape[0]*100,2) ,"%)")

In [ ]:
user_simple_table.sort_values(by='point', ascending=False)

In [ ]:
sns.boxplot(x=user_simple_table['point'])
plt.title('Point 분포 확인 (이상치 탐색)')
plt.show()

포인트 구매이력과 비교

In [ ]:
positive_point_users = user_simple_table[user_simple_table['point'] > 0]
zero_point_users = user_simple_table[user_simple_table['point'] == 0]
print("포인트가 있는 유저 수:", len(positive_point_users))
print("포인트가 0인 유저 수:", len(zero_point_users))

In [ ]:
paymenthistory_df.head()

In [ ]:
paymenthistory_df['user_id'].nunique()

학교_학년별 비교

In [ ]:
user_df['school_grade'] = user_df['school_type'] + '_' + user_df['grade'].astype('str')

In [ ]:
user_school = user_df[['user_id', 'school_grade']]
user_school

In [ ]:
merged_df = vote_point_df[['user_id', 'question_id', 'delta_point', 'created_at']].merge(user_school, how='left', on='user_id')
merged_df

In [ ]:
merged_df.groupby('school_grade')['delta_point'].agg(['count', 'mean'])

### 1-3) 친구 수 & 출석일 반영 후 상관관계 분석(user_ver2 생성)

In [ ]:
correlation = user_df[['attendance_count', 'friend_count', 'block_user_id_count', 'hide_user_id_count', 'point', 'report_count',	'alarm_count',	'pending_chat',	'pending_votes']].corr()

plt.figure(figsize=(7,7))
sns.heatmap(correlation, annot=True, cmap='Spectral', fmt=".2f", square=False)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
user_df['ban_status'].value_counts()

### 2) 투표 참여 유저 비율 확인

In [ ]:
user_cnt = user_df['user_id'].nunique()
chosen_user_cnt = vote_point_df['chosen_user_id'].nunique()
vote_user_cnt = vote_point_df['user_id'].nunique()

In [ ]:
print("전체 유저수 : ", user_cnt)
print("선택된 유저수 : ", chosen_user_cnt, "(", round(chosen_user_cnt/user_cnt*100,3) , "%)")
print("선택한 유저수 : ", vote_user_cnt, "(", round(vote_user_cnt/user_cnt*100, 3), "%)")

In [ ]:
user_df['user_id'].nunique()

In [ ]:
vote_point_df.tail(20)

In [ ]:
vote_point_df.dtypes

In [ ]:
chosen_set = set(vote_point_df['chosen_user_id'].dropna().astype(int))
user_set = set(vote_point_df['user_id'].dropna().astype(int))

In [ ]:
# 2. 겹치는 사람 수
intersection_count = len(chosen_set & user_set)
print("겹치는 유저수 : ", intersection_count)

# 3. 안 겹치는 사람 수
# - chosen_user_id에만 있는 사람 수
only_chosen = len(chosen_set - user_set)
print("선택되기만 한 유저수 : ", only_chosen)

# - user_id에만 있는 사람 수
only_user = len(user_set - chosen_set)
print("선택하기만 한 유저수 : ", only_user)

### 3) 포인트 사용내역 확인

In [ ]:
pointhistory_df.head()

In [ ]:
pointhistory_df.query('delta_point<0')

In [ ]:
sns.kdeplot(pointhistory_df['delta_point'])
plt.title('delta point의 kdeplot')
plt.show()

In [ ]:
sns.kdeplot(vote_point_df['delta_point'])
plt.title('delta point의 kdeplot')
plt.show()

In [ ]:
vote_point_df[vote_point_df['delta_point'] >= 0]

In [ ]:
vote_point_df.groupby('status')['delta_point'].agg(['count', 'mean'])

In [ ]:
vote_point_df.groupby('has_read')['delta_point'].agg(['count', 'mean'])

In [ ]:
vote_point_df.groupby('answer_status')['delta_point'].agg(['count', 'mean'])

In [ ]:
vote_point_df.groupby('opened_times')['delta_point'].agg(['count', 'mean'])

In [ ]:
vote_point_df.groupby(['opened_times','has_read'])['delta_point'].agg(['count', 'mean'])

In [ ]:
vote_simple_table = mem_con.execute("""
                            SELECT user_id,	STRFTIME(created_at, '%Y_%m') AS year_month, question_id,	chosen_user_id,	delta_point, has_read, answer_status, report_count, opened_times,
                            FROM vote_point_df
                            """).df()
vote_simple_table.head()

In [ ]:
vote_date = vote_simple_table['year_month'].value_counts().reset_index()
vote_date

In [ ]:
sns.lineplot(data= vote_date, x='year_month', y='count')
plt.xticks(rotation=45)
plt.title("날짜별 투표 참여이력 수")
plt.show()

### 4) 질문내용 확인

In [ ]:
vote_question_table = mem_con.execute("""
    SELECT 
        v.user_id,
        STRFTIME(v.created_at, '%Y_%m') AS year_month,
        v.question_id,
        q.question_text,
        v.chosen_user_id,
        v.delta_point,
        v.has_read,
        v.answer_status,
        v.report_count,
        v.opened_times
    FROM vote_point_df AS v
    LEFT JOIN polls_question_df AS q
    ON v.question_id = q.id
""").df()

In [ ]:
vote_question_table.head()

In [ ]:
vote_question_table['question_text'].nunique()

In [ ]:
print("상위 10개 질문", vote_question_table['question_text'].value_counts().head(10))
print('-------------------------')
print("하위 10개 질문", vote_question_table['question_text'].value_counts().tail(10))

### 5) 이벤트 테이블

In [ ]:
event_info_df.head()

In [ ]:
event_info_df['user_id'].nunique()

### 6) point 분석

In [ ]:
vote_point_df_ver2.head()

In [ ]:
vote_point_df_ver2.shape

In [ ]:
print("선택된 유저수 : ", vote_point_df_ver2['chosen_user_id'].nunique())
print("선택한 유저수 : ", vote_point_df_ver2['user_id_x'].nunique())

In [ ]:
vote_point_df_ver2.isna().sum()

In [ ]:
vote_point_df_ver2[vote_point_df_ver2['delta_point'].isna()]

테이블 나누기

In [ ]:
chooser_df = vote_point_df_ver2.loc[vote_point_df_ver2['point_user'] == 'chooser']
chooser_df.head()

In [ ]:
chooser_df['user_id_x'].nunique()

In [ ]:
print(chooser_df['delta_point'].mean())
print(chooser_df['delta_point'].min())

In [ ]:
chooser_df['delta_point'].nunique()

In [ ]:
chooser_df['delta_point'].unique()

In [ ]:
chosen_df = vote_point_df_ver2.loc[vote_point_df_ver2['point_user'] == 'chosen']
chosen_df.head()

In [ ]:
chosen_df['chosen_user_id'].nunique()

In [ ]:
chosen_df['delta_point'].nunique()

In [ ]:
chosen_df['delta_point'].unique()

### 7) 탈퇴기록

In [ ]:
userwithdraw_df.head()

In [ ]:
userwithdraw_df.shape

In [ ]:
userwithdraw_df['year'] = userwithdraw_df['created_at'].dt.year
userwithdraw_df['month'] = userwithdraw_df['created_at'].dt.month
userwithdraw_df['year_month'] = userwithdraw_df['year'].astype('str') + '_' + userwithdraw_df['month'].astype('str')

In [ ]:
userwithdraw_df['year_month'].value_counts().reset_index()

In [ ]:
userwithdraw_df['reason'].unique()

In [ ]:
reason_df = userwithdraw_df['reason'].value_counts().reset_index()
reason_df['percentage'] = round((reason_df['count']/userwithdraw_df.shape[0])*100, 3)
reason_df

# 4. 분류모델 돌려보기

In [ ]:
final_user_df = mem_con.execute("""
                FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/final_user_df.parquet')
                """).df()

In [ ]:
final_user_df.head()

In [ ]:
final_user_df.describe()

### 0-0) 전처리 - year_month 생성

In [ ]:
final_user_df['year'] = final_user_df['created_at'].dt.year
final_user_df['month'] = final_user_df['created_at'].dt.month
final_user_df['year_month'] = final_user_df['year'].astype('str') + '_' + final_user_df['month'].astype('str')

In [ ]:
print(final_user_df.shape)
print(final_user_df['user_id'].nunique())

In [ ]:
final_user_df.describe()

### 0-1) province 지역 컬럼 생성

In [ ]:
import re

In [ ]:
addresses  = final_user_df['address'].unique()

# 전체 시도 이름 + 줄임말 모두 대응
province_mapping = {
    '서울': '서울특별시', '서울특별시': '서울특별시',
    '부산': '부산광역시', '부산광역시': '부산광역시',
    '대구': '대구광역시', '대구광역시': '대구광역시',
    '인천': '인천광역시', '인천광역시': '인천광역시',
    '광주': '광주광역시', '광주광역시': '광주광역시',
    '대전': '대전광역시', '대전광역시': '대전광역시',
    '울산': '울산광역시', '울산광역시': '울산광역시',
    '세종': '세종특별자치시', '세종특별자치시': '세종특별자치시',
    '경기': '경기도', '경기도': '경기도',
    '강원': '강원도', '강원도': '강원도',
    '충북': '충청북도', '충청북도': '충청북도',
    '충남': '충청남도', '충청남도': '충청남도',
    '전북': '전라북도', '전라북도': '전라북도',
    '전남': '전라남도', '전라남도': '전라남도',
    '경북': '경상북도', '경상북도': '경상북도',
    '경남': '경상남도', '경상남도': '경상남도',
    '제주': '제주특별자치도', '제주특별자치도': '제주특별자치도'
}

def extract_province(address):
    for short, full in province_mapping.items():
        if short in address:
            return full
    return None

# 적용
final_user_df['province'] = final_user_df['address'].apply(extract_province)



In [ ]:
final_user_df['province'].unique()

In [ ]:
final_user_df[final_user_df['province'].isna()]

In [ ]:
final_user_df.isna().sum()

지역 null값 3건 삭제처리

In [ ]:
final_user_df.dropna(inplace=True)
final_user_df.shape

### 0-2) 전처리 - pending_chat 음수 제거(5개)

In [ ]:
# pending_chat 음수 제거
final_user_df = final_user_df[final_user_df['pending_chat'] >= 0]

In [ ]:
final_user_df.shape

### 0-3) 상관관계 보기

In [ ]:
# 숫자형 컬럼만 선택
numeric_cols = final_user_df.select_dtypes(include=['number']).columns
exclude_cols = ['user_id', 'is_push_on']
f_numeric_cols = [col for col in numeric_cols if col not in exclude_cols]
f_numeric_cols

In [ ]:
corr = final_user_df[f_numeric_cols].corr()

plt.figure(figsize=(10,10))
sns.heatmap(corr, annot=True, cbar=False, fmt=".2f")
plt.show()

### 0-4) 미사용자 제외

In [ ]:
final_user_df.head()

vote_cnt == 0, chosen_cnt == 0, event_cnt == 0, pending_chat = 0, attendance_count == 0 인 유저 찾기

In [ ]:
final_user_df[
    (final_user_df['vote_cnt'] == 0) &
    (final_user_df['chosen_cnt'] == 0) &
    (final_user_df['event_cnt'] == 0) &
    (final_user_df['pending_chat'] == 0) &
    (final_user_df['attendance_count'] == 0)
].shape

In [ ]:
filtered_df = final_user_df[
    ~(
        (final_user_df['vote_cnt'] == 0) &
        (final_user_df['chosen_cnt'] == 0) &
        (final_user_df['event_cnt'] == 0) &
        (final_user_df['pending_chat'] == 0) &
        (final_user_df['attendance_count'] == 0)
    )
]

In [ ]:
filtered_df.shape

In [ ]:
final_user_df = filtered_df.copy()

### 0-4) 이상치 제거

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(final_user_df[f_numeric_cols])
plt.xticks(rotation=90)
plt.show()

In [ ]:
final_user_df.shape

In [ ]:
def remove_iqr(df, col):
  q1 = df[col].quantile(0.25)
  q3 = df[col].quantile(0.75)
  iqr = q3 - q1
  
  lower_bound = q1 - 1.5*iqr
  upper_bound = q3 + 1.5*iqr
  
  df_cleand = df[(df[col]>=lower_bound) & (df[col]<=upper_bound)]
  
  return df_cleand

In [ ]:
final_user_df = remove_iqr(final_user_df, 'point')

In [ ]:
final_user_df.shape

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(final_user_df[f_numeric_cols])
plt.xticks(rotation=90)
plt.show()

### [final_user_df 최종 버전 db에 저장]

In [ ]:
mem_con.execute("DROP TABLE final_user_df")

In [ ]:
mem_con.execute("""
                CREATE TABLE final_user_df AS
                SELECT * FROM final_user_df
                """)

# 다음에 불러올 때 db에서 불러오기

In [ ]:
tables = mem_con.execute("SHOW TABLES").fetchall()
print(tables)

### 1) skewed 처리

In [ ]:
final_user_df = mem_con.execute("SELECT * FROM final_user_df").df()

In [ ]:
cols = [
    'point', 'report_count', 'alarm_count', 'friend_count',
    'block_user_id_count', 'hide_user_id_count', 'attendance_count',
    'pending_chat', 'pending_votes', 'student_count', 'pointuse_cnt',
    'pointget_cnt', 'vote_cnt', 'chosen_cnt', 'event_cnt', 'buy_cnt',
    'buyfail_cnt', 'friendreport_cnt'
]

In [ ]:
import math

rows = math.ceil(len(cols) / 3)
fig, axes = plt.subplots(rows, 3, figsize=(15, 5*rows))
axes = axes.flatten()

for i, col in enumerate(cols):
    sns.histplot(final_user_df[col].dropna(), kde=False, ax=axes[i], bins=50)
    axes[i].set_title(f'Distribution of {col}')

for j in range(len(cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
skewness = final_user_df[cols].skew()
print(skewness)

In [ ]:
skew_threshold = 1.5
skewed_cols = final_user_df[cols].skew()
skewed_cols = skewed_cols[skewed_cols.abs() > skew_threshold].index

noskewed_df = final_user_df.copy()
noskewed_df[skewed_cols] = final_user_df[skewed_cols].apply(np.log1p)

In [ ]:
import math

rows = math.ceil(len(cols) / 3)
fig, axes = plt.subplots(rows, 3, figsize=(15, 5*rows))
axes = axes.flatten()

for i, col in enumerate(cols):
    sns.histplot(noskewed_df[col].dropna(), kde=False, ax=axes[i], bins=50)
    axes[i].set_title(f'Distribution of {col}')

for j in range(len(cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### 2) 인코딩

In [ ]:
noskewed_df.columns

In [ ]:
cols = ['gender', 'ban_status', 'school_grade', 'province', 'year_month']

for col in cols:
  print(noskewed_df[col].unique())

In [ ]:
encoded_df = pd.get_dummies(noskewed_df, columns=cols, drop_first=False, dtype=int)

In [ ]:
encoded_df.head(1)

### 3) 스케일링

In [ ]:
# 세팅하기
from sklearn.preprocessing import StandardScaler

In [ ]:
encoded_df.columns

In [ ]:
# 연속형 변수 리스트
con_cols = ['point','report_count',
       'alarm_count', 'friend_count', 'block_user_id_count',
       'hide_user_id_count', 'attendance_count', 'pending_chat',
       'pending_votes',  'student_count', 'pointuse_cnt', 'pointget_cnt', 'vote_cnt', 'chosen_cnt', 'event_cnt',
       'buy_cnt', 'buyfail_cnt', 'friendreport_cnt']

scaler = StandardScaler()
df_scaled = encoded_df.copy()
df_scaled[con_cols] = scaler.fit_transform(encoded_df[con_cols])

In [ ]:
# 주기형 변수 변환
df_scaled['month_sin'] = np.sin(2 * np.pi * df_scaled['month'] / 12)
df_scaled['month_cos'] = np.cos(2 * np.pi * df_scaled['month'] / 12)

### 4) 주성분분석

In [ ]:
# 세팅하기

from sklearn.decomposition import PCA

In [ ]:
df_scaled.columns

제외
       'year_month_2023_10', 'year_month_2023_11', 'year_month_2023_12',
       'year_month_2023_3', 'year_month_2023_4', 'year_month_2023_5',
       'year_month_2023_6', 'year_month_2023_7', 'year_month_2023_8',
       'year_month_2023_9', 'year_month_2024_1', 'year_month_2024_2',
       'year_month_2024_3', 'year_month_2024_4', 'year_month_2024_5'
       
       'event_cnt', 'buyfail_cnt',
       
        'province_강원도', 'province_경기도', 'province_경상남도',
       'province_경상북도', 'province_광주광역시', 'province_대구광역시', 'province_대전광역시',
       'province_부산광역시', 'province_서울특별시', 'province_세종특별자치시',
       'province_울산광역시', 'province_인천광역시', 'province_전라남도', 'province_전라북도',
       'province_제주특별자치도', 'province_충청남도', 'province_충청북도'
       

In [ ]:
num_cols = ['point', 'is_push_on', 'report_count', 'alarm_count', 'friend_count', 
       'block_user_id_count','hide_user_id_count', 'attendance_count', 
       'pending_chat','pending_votes', 'student_count', 'pointuse_cnt',
       'pointget_cnt', 'vote_cnt', 'chosen_cnt',  'buy_cnt',
       'friendreport_cnt','gender_F','gender_M', 
       'ban_status_N', 'ban_status_NB', 'ban_status_RB',
       'ban_status_W', 'school_grade_H_1', 'school_grade_H_2',
       'school_grade_H_3', 'school_grade_M_1', 'school_grade_M_2',
       'school_grade_M_3','year', 'month_sin', 'month_cos',
       ]

scaled_df = df_scaled[num_cols]

In [ ]:
pca = PCA()
scaled_df_pca = pca.fit_transform(scaled_df)
pca_df = pd.DataFrame(scaled_df_pca)

In [ ]:
pca_df.head()

In [ ]:
pca.explained_variance_ratio_ # 전체 분산 대비 분산 비율 출력

In [ ]:
# Scree Plot그리기
num_components = len(pca.explained_variance_ratio_)

x = np.arange(num_components)
var = pca.explained_variance_ratio_

# 시각화
plt.figure(figsize=(7,4))
plt.bar(x, var)
plt.xlabel('PC')
plt.ylabel('Variance Ratio')
plt.title('Scree Plot')
plt.show()

In [ ]:
# 누적 분산 비율 구하기
cum_var = np.cumsum(var) # np.cumsum : 누적합 구하는 함수
cum_vars = pd.DataFrame({'cum_var' : cum_var}, index=pca_df.columns)
cum_vars.head(15)

In [ ]:
pca = PCA(n_components=8)
scaled_df_pca = pca.fit_transform(scaled_df)
pca_df = pd.DataFrame(scaled_df_pca)

### 5) kmeans

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
import warnings
warnings.filterwarnings('ignore')

inertias = []

for k in range(1, 16):
    # 여기에 코드를 작성하세요
    model = KMeans(n_clusters=k, random_state=45)
    model.fit(pca_df)
    inertias.append(model.inertia_)


# 시각화
sns.set(style="darkgrid")
sns.lineplot(x=range(1,16), y=inertias, marker='o')

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=45)

kmeans.fit(pca_df)

In [ ]:
labels = kmeans.predict(pca_df)
labels

In [ ]:
final_user_df_label = final_user_df.copy()
final_user_df_label['kmeans_labels'] = labels

In [ ]:
final_user_df_label['kmeans_labels'].value_counts()

In [ ]:
df = final_user_df_label[['point','report_count', 'alarm_count', 'friend_count', 'block_user_id_count',
                          'hide_user_id_count', 'attendance_count', 'pending_chat', 'pending_votes',  
                          'student_count', 'pointuse_cnt', 'pointget_cnt', 'vote_cnt', 'chosen_cnt', 'event_cnt',
                          'buy_cnt', 'buyfail_cnt', 'friendreport_cnt', 'kmeans_labels']]

In [ ]:
df.groupby('kmeans_labels').mean().T.style.format('{:.2f}')

In [ ]:
final_user_df_label.groupby('kmeans_labels')['ban_status'].value_counts().T.reset_index()

In [ ]:
from sklearn.metrics import silhouette_score

# 67만 개의 데이터와 레이블이 있다고 가정
# pca_df = ...
# labels = ...

# 전체 데이터 중 10,000개를 무작위로 샘플링
sample_size = 50000 
random_indices = np.random.choice(pca_df.shape[0], size=sample_size, replace=False)

# 샘플링된 데이터와 레이블 추출
sampled_pca_df = pca_df.iloc[random_indices]
sampled_labels = labels[random_indices]

# 샘플링된 데이터로 실루엣 점수 계산
silhouette = silhouette_score(sampled_pca_df, sampled_labels)
print("Silhouette Score (sampled):", silhouette)

### 5. 친구수의 영향

In [ ]:
final_user_df = mem_con.execute("SELECT * FROM final_user_df").df()

In [ ]:
final_user_df.head()

In [ ]:
final_user_df.shape

In [ ]:
final_user_df['friend_count'].describe()

In [ ]:
# Step 1: 친구수 구간 설정하기

# 먼저 친구수 분포 확인
print("=== 친구수 기본 통계 ===")
print(final_user_df['friend_count'].describe())

# 방법 1: 분위수로 4구간 나누기 (가장 간단)
final_user_df['friend_group_quartile'] = pd.qcut(final_user_df['friend_count'], 
                                                q=4, 
                                                labels=['1구간(하위25%)', '2구간(25-50%)', 
                                                       '3구간(50-75%)', '4구간(상위25%)'])

print("\n=== 방법 1: 분위수 4구간 결과 ===")
print(final_user_df['friend_group_quartile'].value_counts().sort_index())

In [ ]:
# 구간별 통계 계산
range_stats = final_user_df.groupby('friend_group_quartile')['friend_count'].agg([
    'min', 'max', 'mean', 'count'
]).round(1)

# 컬럼명 한글로 변경
range_stats.columns = ['최소값', '최대값', '평균', '사용자수']

print("=== 친구수 구간별 범위 ===")
print(range_stats)

# 더 깔끔한 버전 (범위 표시)
print("\n=== 구간별 범위 요약 ===")
for idx, (group_name, stats) in enumerate(range_stats.iterrows()):
    min_val = int(stats['최소값'])
    max_val = int(stats['최대값'])
    mean_val = stats['평균']
    count_val = int(stats['사용자수'])
    
    print(f"{group_name}: {min_val}~{max_val}명 (평균 {mean_val:.1f}명, {count_val:,}명)")

In [ ]:
# Step 2: 구간별 특징 비교 분석

# 분석할 주요 컬럼들 선택
key_columns = [
    'friend_count',      # 친구수 (기준)
    'point',            # 포인트
    'attendance_count', # 출석 횟수  
    'vote_cnt',         # 투표 참여
    'chosen_cnt',        # 투표에서 선택됨
    'buy_cnt',          # 구매 횟수
    'alarm_count',      # 알람 수
    'pointuse_cnt',     # 포인트 사용 횟수
    'pointget_cnt'      # 포인트 획득 횟수
]

# 구간별 평균값 비교
print("=== 구간별 주요 지표 평균값 ===")
group_means = final_user_df.groupby('friend_group_quartile')[key_columns].mean().round(2)
print(group_means)

print("\n=== 구간별 주요 지표 중앙값 ===")
group_medians = final_user_df.groupby('friend_group_quartile')[key_columns].median().round(2)
print(group_medians)


In [ ]:
# Step 3-1: ANOVA 가정 검정

from scipy import stats
import pandas as pd
import numpy as np

# 검정할 지표들
test_metrics = ['vote_cnt', 'buy_cnt', 'chosen_cnt', 'pointuse_cnt', 'pointget_cnt']

print("="*70)
print("=== ANOVA 가정 검정 ===")
print("="*70)

assumption_results = {}

for metric in test_metrics:
    print(f"\n🔍 {metric} 검정:")
    print("-" * 40)
    
    assumption_results[metric] = {
        'normality_violated': False,
        'homogeneity_violated': False,
        'recommended_test': 'ANOVA'
    }
    
    # 1. 정규성 검정 (Shapiro-Wilk)
    print("📊 1. 정규성 검정 (Shapiro-Wilk test)")
    print("   H0: 데이터가 정규분포를 따른다")
    print("   H1: 데이터가 정규분포를 따르지 않는다")
    
    normality_violated = False
    
    for group in final_user_df['friend_group_quartile'].cat.categories:
        group_data = final_user_df[final_user_df['friend_group_quartile'] == group][metric].dropna()
        
        # 샘플이 너무 크면 shapiro 검정이 안 되므로 sampling
        if len(group_data) > 5000:
            group_data = group_data.sample(5000, random_state=42)
        
        if len(group_data) >= 3:  # 최소 샘플 수 확인
            shapiro_stat, shapiro_p = stats.shapiro(group_data)
            
            print(f"   {group}: W = {shapiro_stat:.4f}, p = {shapiro_p:.6f}", end="")
            
            if shapiro_p < 0.05:
                print(" ❌ 비정규")
                normality_violated = True
            else:
                print(" ✅ 정규")
    
    assumption_results[metric]['normality_violated'] = normality_violated
    
    # 2. 등분산성 검정 (Levene's test)
    print(f"\n📊 2. 등분산성 검정 (Levene's test)")
    print("   H0: 모든 그룹의 분산이 같다")
    print("   H1: 적어도 하나의 그룹 분산이 다르다")
    
    groups_for_levene = []
    for group in final_user_df['friend_group_quartile'].cat.categories:
        group_data = final_user_df[final_user_df['friend_group_quartile'] == group][metric].dropna()
        groups_for_levene.append(group_data)
    
    levene_stat, levene_p = stats.levene(*groups_for_levene)
    
    print(f"   Levene 통계량: {levene_stat:.4f}")
    print(f"   p값: {levene_p:.6f}")
    
    if levene_p < 0.05:
        print("   ❌ 등분산성 가정 위반")
        assumption_results[metric]['homogeneity_violated'] = True
    else:
        print("   ✅ 등분산성 가정 만족")
    
    # 3. 추천 검정 방법 결정
    print(f"\n💡 추천 검정 방법:")
    
    if normality_violated and assumption_results[metric]['homogeneity_violated']:
        print("   → Kruskal-Wallis 검정 (비모수) - 두 가정 모두 위반")
        assumption_results[metric]['recommended_test'] = 'Kruskal-Wallis'
    elif normality_violated:
        print("   → Kruskal-Wallis 검정 (비모수) - 정규성 위반")
        assumption_results[metric]['recommended_test'] = 'Kruskal-Wallis'
    elif assumption_results[metric]['homogeneity_violated']:
        print("   → Welch's ANOVA - 등분산성 위반")
        assumption_results[metric]['recommended_test'] = 'Welch ANOVA'
    else:
        print("   → 일반 ANOVA - 모든 가정 만족")
        assumption_results[metric]['recommended_test'] = 'ANOVA'

# 가정 검정 결과 요약
print("\n" + "="*70)
print("=== 가정 검정 결과 요약 ===")
print("="*70)

summary_df = pd.DataFrame(assumption_results).T
print(summary_df)

# 각 검정 방법별 실행
print(f"\n" + "="*70)
print("=== 적절한 검정 방법으로 분석 실행 ===")
print("="*70)

final_test_results = {}

for metric in test_metrics:
    print(f"\n🔍 {metric} - {assumption_results[metric]['recommended_test']} 검정:")
    
    if assumption_results[metric]['recommended_test'] == 'Kruskal-Wallis':
        # 비모수 검정
        groups = []
        for group in final_user_df['friend_group_quartile'].cat.categories:
            group_data = final_user_df[final_user_df['friend_group_quartile'] == group][metric].dropna()
            groups.append(group_data)
        
        h_stat, p_value = stats.kruskal(*groups)
        print(f"   H 통계량: {h_stat:.3f}")
        print(f"   p값: {p_value:.6f}")
        
        final_test_results[metric] = {
            'test_type': 'Kruskal-Wallis',
            'statistic': h_stat,
            'p_value': p_value,
            'significant': p_value < 0.05
        }
        
    elif assumption_results[metric]['recommended_test'] == 'Welch ANOVA':
        # Welch's ANOVA (등분산성 가정 불필요)
        groups = []
        for group in final_user_df['friend_group_quartile'].cat.categories:
            group_data = final_user_df[final_user_df['friend_group_quartile'] == group][metric].dropna()
            groups.append(group_data)
        
        # scipy에는 welch anova가 없어서 일반 ANOVA로 대체하고 주의사항 표시
        f_stat, p_value = stats.f_oneway(*groups)
        print(f"   F 통계량: {f_stat:.3f} (주의: 등분산성 위반)")
        print(f"   p값: {p_value:.6f}")
        print("   ⚠️  등분산성 위반으로 결과 해석에 주의 필요")
        
        final_test_results[metric] = {
            'test_type': 'ANOVA (등분산성 위반)',
            'statistic': f_stat,
            'p_value': p_value,
            'significant': p_value < 0.05
        }
        
    else:
        # 일반 ANOVA
        groups = []
        for group in final_user_df['friend_group_quartile'].cat.categories:
            group_data = final_user_df[final_user_df['friend_group_quartile'] == group][metric].dropna()
            groups.append(group_data)
        
        f_stat, p_value = stats.f_oneway(*groups)
        print(f"   F 통계량: {f_stat:.3f}")
        print(f"   p값: {p_value:.6f}")
        
        final_test_results[metric] = {
            'test_type': 'ANOVA',
            'statistic': f_stat,
            'p_value': p_value,
            'significant': p_value < 0.05
        }
    
    # 결과 해석
    if final_test_results[metric]['significant']:
        print("   ✅ 구간 간 유의한 차이 있음 (p < 0.05)")
    else:
        print("   ❌ 구간 간 유의한 차이 없음 (p >= 0.05)")

# 최종 결과 요약
print(f"\n" + "="*70)
print("=== 최종 검정 결과 ===")
print("="*70)

final_results_df = pd.DataFrame(final_test_results).T
print(final_results_df)

significant_metrics = [metric for metric in test_metrics if final_test_results[metric]['significant']]
print(f"\n✅ 구간 간 유의한 차이가 있는 지표: {len(significant_metrics)}개")
for metric in significant_metrics:
    test_type = final_test_results[metric]['test_type']
    print(f"   • {metric} ({test_type})")

if significant_metrics:
    print(f"\n💡 다음 단계: 위 지표들에 대해 사후검정(Post-hoc test) 실행 가능")